# 0. Exploration des Données Brutes 

**Objectif** : Analyser en profondeur les trois fichiers bruts issus du scraping avant tout nettoyage ou feature engineering.

**Fichiers explorés** :
- `data/raw/players/nba_players_totals_2010_2025.csv`
- `data/raw/teams/nba_teams.csv`
- `data/raw/boxscores/nba_boxscores_2010_2025.csv`


Ce notebook permet de :
- Vérifier la structure et la qualité des données scrapées
- Identifier les problèmes à corriger (valeurs manquantes, formats, doublons, etc.)
- Justifier les étapes de nettoyage ultérieures

In [ ]:
# Import des librairies et configuration
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Configuration d'affichage pandas pour mieux voir les données
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Librairies chargées et configuration prête")

Librairies chargées et configuration prête


In [ ]:
# Définition des chemins et chargement des données brutes
BASE_DIR = Path.cwd().parent  # Adapté si le notebook est dans un sous-dossier notebooks/

RAW_PLAYERS   = BASE_DIR / "data" / "raw" / "players" / "nba_players_totals_2010_2025.csv"
RAW_TEAMS     = BASE_DIR / "data" / "raw" / "teams" / "nba_teams.csv"
RAW_BOXSCORES = BASE_DIR / "data" / "raw" / "boxscores" / "nba_boxscores_2010_2025.csv"

# Chargement avec vérification d'existence
def load_csv(path, name):
    if path.exists():
        df = pd.read_csv(path)
        print(f"✅ {name} chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
        return df
    else:
        print(f"❌ Fichier manquant : {path}")
        return None

df_players   = load_csv(RAW_PLAYERS, "Joueurs (players_totals)")
df_teams     = load_csv(RAW_TEAMS, "Équipes historiques")
df_boxscores = load_csv(RAW_BOXSCORES, "Boxscores (matchs)")

print("\nChargement terminé.")

✅ Joueurs (players_totals) chargé : 9,731 lignes × 31 colonnes
✅ Équipes historiques chargé : 85 lignes × 13 colonnes
✅ Boxscores (matchs) chargé : 11,197 lignes × 5 colonnes

Chargement terminé.


In [ ]:
# Aperçu visuel rapide
print("=== 5 PREMIÈRES LIGNES - JOUEURS ===")
display(df_players.head())

print("\n=== 5 PREMIÈRES LIGNES - ÉQUIPES ===")
display(df_teams.head())

print("\n=== 5 PREMIÈRES LIGNES - BOXSCORES ===")
display(df_boxscores.head())

print("\n=== 5 DERNIÈRES LIGNES - BOXSCORES (pour voir la fin) ===")
display(df_boxscores.tail())

=== 5 PREMIÈRES LIGNES - JOUEURS ===


,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl,Awards
0,Kevin Durant,21.00,OKC,SF,82.00,82.00,3239.00,794.00,1668.00,0.48,128.00,351.00,0.36,666.00,1317.00,0.51,0.51,756.00,840.00,0.90,105.00,518.00,623.00,231.00,112.00,84.00,271.00,171.00,2472.00,0.00,"MVP-2,AS,NBA1"
1,LeBron James,25.00,CLE,SF,76.00,76.00,2966.00,768.00,1528.00,0.50,129.00,387.00,0.33,639.00,1141.00,0.56,0.55,593.00,773.00,0.77,71.00,483.00,554.00,651.00,125.00,77.00,261.00,119.00,2258.00,4.00,"MVP-1,DPOY-4,AS,NBA1,DEF1"
2,Dwyane Wade,28.00,MIA,SG,77.00,77.00,2792.00,719.00,1511.00,0.48,73.00,243.00,0.30,646.00,1268.00,0.51,0.50,534.00,702.00,0.76,107.00,266.00,373.00,501.00,142.00,82.00,252.00,181.00,2045.00,0.00,"MVP-5,DPOY-10,AS,NBA1,DEF2"
3,Dirk Nowitzki,31.00,DAL,PF,81.00,80.00,3039.00,720.00,1496.00,0.48,51.00,121.00,0.42,669.00,1375.00,0.49,0.50,536.00,586.00,0.92,80.00,540.00,620.00,218.00,70.00,79.00,149.00,208.00,2027.00,1.00,"MVP-7,AS,NBA2"
4,Kobe Bryant,31.00,LAL,SG,73.00,73.00,2835.00,716.00,1569.00,0.46,99.00,301.00,0.33,617.00,1268.00,0.49,0.49,439.00,541.00,0.81,78.00,313.00,391.00,365.00,113.00,20.00,233.00,187.00,1970.00,0.00,"MVP-3,DPOY-12,AS,NBA1,DEF1"



=== 5 PREMIÈRES LIGNES - ÉQUIPES ===


,Franchise,Lg,From,To,Yrs,G,W,L,W/L%,Plyfs,Div,Conf,Champ
0,Atlanta Hawks,NBA,1949-50,2025-26,77,6039,2978,3060,0.49,49,12,0,1
1,Atlanta Hawks,NBA,1968-69,2025-26,58,4620,2280,2340,0.49,36,6,0,0
2,St. Louis Hawks,NBA,1955-56,1967-68,13,1005,553,452,0.55,12,6,0,1
3,Milwaukee Hawks,NBA,1951-52,1954-55,4,282,91,190,0.32,0,0,0,0
4,Tri-Cities Blackhawks,NBA,1949-50,1950-51,2,132,54,78,0.41,1,0,0,0



=== 5 PREMIÈRES LIGNES - BOXSCORES ===


,Date,AwayTeam,HomeTeam,AwayPts,HomePts
0,"Tue, Oct 26, 2010",Miami Heat,Boston Celtics,80,88
1,"Tue, Oct 26, 2010",Phoenix Suns,Portland Trail Blazers,92,106
2,"Tue, Oct 26, 2010",Houston Rockets,Los Angeles Lakers,110,112
3,"Wed, Oct 27, 2010",Boston Celtics,Cleveland Cavaliers,87,95
4,"Wed, Oct 27, 2010",New York Knicks,Toronto Raptors,98,93



=== 5 DERNIÈRES LIGNES - BOXSCORES (pour voir la fin) ===


,Date,AwayTeam,HomeTeam,AwayPts,HomePts
11192,"Mon, Apr 29, 2024",Oklahoma City Thunder,New Orleans Pelicans,97,89
11193,"Mon, Apr 29, 2024",Los Angeles Lakers,Denver Nuggets,106,108
11194,"Tue, Apr 30, 2024",Philadelphia 76ers,New York Knicks,112,106
11195,"Tue, Apr 30, 2024",Orlando Magic,Cleveland Cavaliers,103,104
11196,"Tue, Apr 30, 2024",Indiana Pacers,Milwaukee Bucks,92,115


In [4]:
#  Informations globales
def info_dataset(df, nom):
    print(f"\n=== INFO - {nom.upper()} ===")
    print(f"Forme            : {df.shape}")
    print(f"Mémoire utilisée : {df.memory_usage(deep=True).sum() / 1024**2:.2f} Mo")
    print(f"Types de colonnes :\n{df.dtypes.value_counts()}")
    print("\nTypes détaillés :")
    print(df.dtypes)

info_dataset(df_players, "Joueurs")
info_dataset(df_teams, "Équipes")
info_dataset(df_boxscores, "Boxscores")


=== INFO - JOUEURS ===
Forme            : (9731, 31)
Mémoire utilisée : 3.86 Mo
Types de colonnes :
float64    27
object      4
Name: count, dtype: int64

Types détaillés :
Player      object
Age        float64
Team        object
Pos         object
G          float64
GS         float64
MP         float64
FG         float64
FGA        float64
FG%        float64
3P         float64
3PA        float64
3P%        float64
2P         float64
2PA        float64
2P%        float64
eFG%       float64
FT         float64
FTA        float64
FT%        float64
ORB        float64
DRB        float64
TRB        float64
AST        float64
STL        float64
BLK        float64
TOV        float64
PF         float64
PTS        float64
Trp-Dbl    float64
Awards      object
dtype: object

=== INFO - ÉQUIPES ===
Forme            : (85, 13)
Mémoire utilisée : 0.02 Mo
Types de colonnes :
int64      8
object     4
float64    1
Name: count, dtype: int64

Types détaillés :
Franchise     object
Lg            objec

In [ ]:
# Analyse des valeurs manquantes
def analyse_missing(df, nom):
    print(f"\n=== VALEURS MANQUANTES - {nom.upper()} ===")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_table = pd.DataFrame({
        'Nb manquantes': missing,
        'Pourcentage (%)': missing_pct
    }).sort_values('Nb manquantes', ascending=False)
    
    missing_table = missing_table[missing_table['Nb manquantes'] > 0]
    
    if missing_table.empty:
        print("➜ Aucune valeur manquante détectée")
    else:
        display(missing_table.head(20))

analyse_missing(df_players, "Joueurs")
analyse_missing(df_teams, "Équipes")
analyse_missing(df_boxscores, "Boxscores")


=== VALEURS MANQUANTES - JOUEURS ===


,Nb manquantes,Pourcentage (%)
Awards,8840,90.84
3P%,1001,10.29
FT%,558,5.73
2P%,132,1.36
eFG%,64,0.66
FG%,64,0.66
Age,15,0.15
3P,15,0.15
MP,15,0.15
FGA,15,0.15



=== VALEURS MANQUANTES - ÉQUIPES ===
➜ Aucune valeur manquante détectée

=== VALEURS MANQUANTES - BOXSCORES ===
➜ Aucune valeur manquante détectée


In [6]:
# Doublons
print("=== DOUBLONS ===")
print(f"Joueurs   : {df_players.duplicated().sum():,} doublons complets")
print(f"Équipes   : {df_teams.duplicated().sum():,} doublons complets")
print(f"Boxscores : {df_boxscores.duplicated().sum():,} doublons complets")

=== DOUBLONS ===
Joueurs   : 0 doublons complets
Équipes   : 0 doublons complets
Boxscores : 0 doublons complets


In [10]:
# ============================================================
# FOCUS SUR LES PROBLEMES IDENTIFIES (VERSION ROBUSTE)
# ============================================================

# ---------- UTILS ----------
def safe_columns(df, cols):
    """Retourne uniquement les colonnes existantes"""
    return [c for c in cols if c in df.columns]

def detect_team_column(df):
    """Detecte automatiquement la colonne equipe"""
    for c in ['Tm', 'Team', 'Teams', 'Franchise']:
        if c in df.columns:
            return c
    return None


# ============================================================
# JOUEURS
# ============================================================

print("=== JOUEURS - Exemples de problemes connus ===")

# --- Decimales sans zero ---
print("• Decimales sans zero :")
cols_pct = safe_columns(df_players, ['FG%', '3P%', 'FT%'])
if cols_pct:
    display(df_players[cols_pct].head(10))
else:
    print("Aucune colonne % trouvee chez les joueurs")

# --- League Average ---
print("\n• Lignes 'League Average' :")
if 'Player' in df_players.columns:
    display(df_players[df_players['Player'].str.contains('League Average', na=False)].head())
else:
    print("Colonne 'Player' absente")

# --- Joueurs trades (TM) ---
print("\n• Lignes avec 'TM' :")
team_col_players = detect_team_column(df_players)
if team_col_players:
    display(df_players[df_players[team_col_players].astype(str).str.contains('TM', na=False)].head(10))
else:
    print("Aucune colonne equipe detectee chez les joueurs")


# ============================================================
# BOXSCORES
# ============================================================

print("\n=== BOXSCORES - Format des dates ===")
if 'Date' in df_boxscores.columns:
    display(df_boxscores['Date'].head(15))
    print("→ Format observe : 'Tue, Oct 26, 2010' → a convertir en datetime")
else:
    print("Colonne 'Date' absente dans df_boxscores")


# ============================================================
# EQUIPES
# ============================================================

print("\n=== EQUIPES - Problemes de format ===")
cols_teams = safe_columns(df_teams, ['Franchise', 'Team', 'W/L%'])
if cols_teams:
    display(df_teams[cols_teams].head(10))
    print("→ Meme probleme de decimales '.493' → 0.493")
else:
    print("Colonnes equipes non disponibles")


=== JOUEURS - Exemples de problemes connus ===
• Decimales sans zero :


,FG%,3P%,FT%
0,0.48,0.36,0.90
1,0.50,0.33,0.77
2,0.48,0.30,0.76
3,0.48,0.42,0.92
4,0.46,0.33,0.81
5,0.46,0.32,0.83
6,0.56,0.17,0.77
7,0.49,0.29,0.78
8,0.52,0.36,0.80
9,0.42,0.33,0.78



• Lignes 'League Average' :


,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl,Awards
578,League Average,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.46,NaN,NaN,0.35,NaN,NaN,0.49,0.50,NaN,NaN,0.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1204,League Average,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.46,NaN,NaN,0.36,NaN,NaN,0.49,0.50,NaN,NaN,0.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1756,League Average,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.45,NaN,NaN,0.35,NaN,NaN,0.48,0.49,NaN,NaN,0.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2330,League Average,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.45,NaN,NaN,0.36,NaN,NaN,0.48,0.50,NaN,NaN,0.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2942,League Average,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.45,NaN,NaN,0.36,NaN,NaN,0.49,0.50,NaN,NaN,0.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



• Lignes avec 'TM' :


...


=== BOXSCORES - Format des dates ===


0     Tue, Oct 26, 2010
1     Tue, Oct 26, 2010
2     Tue, Oct 26, 2010
3     Wed, Oct 27, 2010
4     Wed, Oct 27, 2010
5     Wed, Oct 27, 2010
6     Wed, Oct 27, 2010
7     Wed, Oct 27, 2010
8     Wed, Oct 27, 2010
9     Wed, Oct 27, 2010
10    Wed, Oct 27, 2010
11    Wed, Oct 27, 2010
12    Wed, Oct 27, 2010
13    Wed, Oct 27, 2010
14    Wed, Oct 27, 2010
Name: Date, dtype: object

→ Format observe : 'Tue, Oct 26, 2010' → a convertir en datetime

=== EQUIPES - Problemes de format ===


,Franchise,W/L%
0,Atlanta Hawks,0.49
1,Atlanta Hawks,0.49
2,St. Louis Hawks,0.55
3,Milwaukee Hawks,0.32
4,Tri-Cities Blackhawks,0.41
5,Boston Celtics,0.60
6,Brooklyn Nets,0.44
7,Brooklyn Nets,0.45
8,New Jersey Nets,0.42
9,New York Nets,0.27


→ Meme probleme de decimales '.493' → 0.493


In [8]:
#  Statistiques numériques
print("=== STATISTIQUES NUMÉRIQUES - JOUEURS ===")
display(df_players.describe().round(2))

print("\n=== STATISTIQUES NUMÉRIQUES - BOXSCORES (points) ===")
# Conversion temporaire pour voir les stats
df_boxscores_temp = df_boxscores.copy()
df_boxscores_temp['AwayPts'] = pd.to_numeric(df_boxscores_temp['AwayPts'], errors='coerce')
df_boxscores_temp['HomePts'] = pd.to_numeric(df_boxscores_temp['HomePts'], errors='coerce')
print(df_boxscores_temp[['AwayPts', 'HomePts']].describe().round(1))

=== STATISTIQUES NUMÉRIQUES - JOUEURS ===


,Age,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl
count,9716.00,9716.00,9716.00,9716.00,9716.00,9716.00,9667.00,9716.00,9716.00,8730.00,9716.00,9716.00,9599.00,9667.00,9716.00,9716.00,9173.00,9716.00,9716.00,9716.00,9716.00,9716.00,9716.00,9716.00,9716.00,9716.00,9716.00
mean,26.31,43.51,20.08,987.68,159.17,347.54,0.44,39.54,110.68,0.30,119.63,236.86,0.49,0.49,70.89,92.62,0.73,42.76,133.23,175.99,94.71,31.02,19.70,55.35,82.29,428.77,0.14
std,4.17,25.67,25.95,817.12,160.32,340.68,0.11,50.96,134.62,0.15,131.49,254.54,0.12,0.11,90.76,113.80,0.15,52.33,133.15,179.08,119.11,30.10,26.97,56.46,65.29,438.70,1.22
min,19.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,23.00,20.00,0.00,241.00,29.00,68.00,0.40,1.00,6.00,0.25,19.00,41.00,0.44,0.46,10.00,14.00,0.67,8.00,28.00,38.00,14.00,7.00,3.00,11.00,23.00,78.00,0.00
50%,26.00,46.00,6.00,805.00,106.50,238.00,0.44,17.00,54.00,0.33,72.00,147.00,0.49,0.50,38.00,52.00,0.76,24.00,96.00,125.00,52.00,23.00,10.00,38.00,71.00,284.00,0.00
75%,29.00,67.00,35.00,1636.00,245.00,536.00,0.49,62.00,176.00,0.38,177.00,351.00,0.55,0.55,97.00,128.00,0.83,57.00,195.00,255.25,126.00,47.00,25.00,81.00,130.00,654.00,0.00
max,43.00,84.00,83.00,3239.00,857.00,1941.00,1.00,402.00,1028.00,1.00,803.00,1459.00,1.00,1.50,756.00,916.00,1.00,440.00,848.00,1247.00,907.00,191.00,269.00,464.00,332.00,2818.00,42.00



=== STATISTIQUES NUMÉRIQUES - BOXSCORES (points) ===
       AwayPts  HomePts
count 11197.00 11197.00
mean    103.30   106.10
std      13.70    13.60
min      56.00    59.00
25%      94.00    97.00
50%     103.00   106.00
75%     112.00   115.00
max     176.00   175.00


## 6. Conclusions et prochaines étapes

**Problèmes identifiés** :
- Décimales mal formatées (`.493` au lieu de `0.493`) → à corriger
- Dates au format texte américain → à parser
- Lignes "League Average" → à utiliser pour créer la colonne `Season`
- Lignes avec `Tm` contenant "TM" → à supprimer (stats dupliquées)
- Quelques valeurs manquantes dans les scores (matchs non joués ou en cours)

**Prochaines étapes** :
1. Notebook de nettoyage (`clean_players.py`, `clean_boxscores.py`, `clean_teams.py`)
2. Création des datasets propres dans `data/cleaned/`
3. Feature engineering
4. Modélisation

Ce notebook constitue la base justifiant toutes les transformations ultérieures.